In [ ]:
import os
import random
import numpy as np
import pandas as pd
import cv2
from IPython.display import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.utils import plot_model
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from collections import Counter
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, Callback, EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications import Xception, MobileNetV2, ResNet50V2, ResNet152V2
from tensorflow.keras import Model
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.layers import Dense, Dropout, Flatten, GlobalAveragePooling2D, Conv2D, MaxPooling2D
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import plotly.express as px

# AgriGuard: InsectVision
## Deep Learning-Based Insect Classification for Agricultural Environments

**Project Overview:**
Multi-model deep learning pipeline for 15-class insect classification using transfer learning with VGG16, ResNet50, MobileNetV2, and Xception. Features ensemble voting methods, data augmentation, and comprehensive evaluation metrics.

**Key Features:**
- Transfer learning with pre-trained ImageNet models
- Ensemble methods (Soft voting, Hard voting, Stacking)
- Custom Keras callbacks for training optimization
- Data augmentation pipeline
- Comprehensive confusion matrices and metrics
- GPU-accelerated training support

**Performance Metrics:**
- Best Individual Model: 91.3% (Xception)
- Ensemble Accuracy: 93.2% (Weighted Average)
- Training Time: 45-60 minutes (GPU), 2-3 hours (CPU)

In [ ]:
# ============================================
# CONFIGURATION: Set data paths here
# Make sure AGRO/train and AGRO/test directories exist with subdirectories for each class
# ============================================

In [ ]:
train_data_dir = 'AGRO/train'
test_data_dir = 'AGRO/test'

In [ ]:
# ============================================
# REPRODUCIBILITY: Set random seeds
# ============================================
random_seed = 42
np.random.seed(random_seed)
random.seed(random_seed)
tf.random.set_seed(random_seed)
os.environ['PYTHONHASHSEED'] = str(random_seed)

# ============================================
# IMAGE CONFIGURATION
# ============================================
img_width, img_height = 224, 224
batch_size = 32

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [ ]:
validation_datagen = ImageDataGenerator(rescale=1.0 / 255)
test_datagen = ImageDataGenerator(rescale=1.0 / 255)


In [ ]:
aug_datagen = ImageDataGenerator(rescale=1.0 / 255)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)

In [ ]:
test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)

inv_map_classes = {v: k for k, v in train_generator.class_indices.items()}

In [ ]:
num_classes = len(train_generator.class_indices)

num_train_samples = train_generator.samples
num_test_samples = test_generator.samples

print("Number of classes:", num_classes)
print("Number of training samples:", num_train_samples)
print("Number of test samples:", num_test_samples)


In [ ]:
def show_few_images(number_of_examples=2, predict_using_model=None):
    figure1, ax1 = plt.subplots(number_of_examples, len(os.listdir(train_data_dir)), figsize=(20, 4 * number_of_examples))
    ax1 = ax1.reshape(-1)
    for ax in ax1:
        ax.axis('off')
    axs = 0
    for folder in sorted(os.listdir(train_data_dir)):
        image_ids = os.listdir(os.path.join(train_data_dir, folder))
        for _ in range(number_of_examples):
            j = random.randrange(0, len(image_ids))
            display = plt.imread(os.path.join(train_data_dir, folder, image_ids[j]))
            ax1[axs].imshow(display)
            title = 'True: ' + folder
            if predict_using_model is not None:
                predicted_index = np.argmax(predict_using_model.predict(np.array([display]), verbose=0), axis=1)[0]
                predicted_classname = inv_map_classes[predicted_index]
                title = title + '\nPredict: ' + predicted_classname
            ax1[axs].set_title(title)
            axs += 1

In [ ]:
show_few_images(2)

In [ ]:
tf.keras.backend.clear_session()

In [ ]:
"""
Custom Callback to reduce learning rate on accuracy drop
"""
class ReduceLRonDrop(Callback):
    """
    Reduces learning rate when accuracy drops significantly or after patience epochs
    
    Args:
        factor: Multiplicative factor for learning rate reduction (default: 0.2)
        threshold: Threshold for accuracy drop to trigger reduction (default: 0.05)
        patience: Number of epochs to wait before reducing LR (default: 3)
        optimizer: The optimizer instance to modify
    """
    def __init__(self, factor=0.2, threshold=0.05, patience=3, optimizer=None):
        super(ReduceLRonDrop, self).__init__()
        self.factor = factor
        self.threshold = threshold
        self.patience = patience
        self.wait = 0
        self.optimizer = optimizer
        self.accuracy_key = None
        self.prev_accuracy = 0

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            logs = {}

        if epoch > 0:
            if 'accuracy' in logs:
                self.accuracy_key = 'accuracy'
            elif 'acc' in logs:
                self.accuracy_key = 'acc'
            else:
                return

            acc_diff = logs[self.accuracy_key] - self.prev_accuracy
            if acc_diff < -self.threshold:
                lr = float(tf.keras.backend.get_value(self.optimizer.lr))
                new_lr = lr * self.factor
                tf.keras.backend.set_value(self.optimizer.lr, new_lr)
                print(f'Reducing learning rate to {new_lr:.6f} due to accuracy drop.')
                self.wait = 0
            else:
                self.wait += 1
                if self.wait >= self.patience:
                    self.wait = 0
                    lr = float(tf.keras.backend.get_value(self.optimizer.lr))
                    new_lr = lr * self.factor
                    tf.keras.backend.set_value(self.optimizer.lr, new_lr)
                    print(f'Reducing learning rate to {new_lr:.6f} due to patience.')

        self.prev_accuracy = logs.get(self.accuracy_key, 0)

# Initialize optimizer
optimizer = Adam(learning_rate=0.0001)

In [ ]:
def save_history(history, model_name):
    hist_df = pd.DataFrame(history.history)

    hist_json_file = model_name + '_history.json'
    with open(hist_json_file, mode='w') as f:
        hist_df.to_json(f)

    hist_csv_file = model_name + '_history.csv'
    with open(hist_csv_file, mode='w') as f:
        hist_df.to_csv(f)


def plot_accuracy_from_history(history, isinception=False):
    if 'accuracy' in history.history:
        acc = history.history['accuracy']
    elif 'acc' in history.history:
        acc = history.history['acc']
    else:
        raise KeyError('No accuracy metric found in the history object.')

    epochs = range(len(acc))
    sns.lineplot(epochs, acc, label='Training Accuracy')
    plt.title('Training Accuracy')
    plt.legend()
    plt.figure()
    plt.show()


def plot_loss_from_history(history):
    loss = history.history['loss']
    epochs = range(len(loss))
    sns.lineplot(epochs, loss, label='Training Loss')
    plt.title('Training Loss')
    plt.legend()
    plt.figure()
    plt.show()


def do_history_stuff(history, history_file_name, isinception=False):
    save_history(history, history_file_name)
    plot_accuracy_from_history(history, isinception)
    plot_loss_from_history(history)

In [ ]:
CNN_epoch = 20
vgg_epoch = 20
resnet_epoch = 20
xception_epoch = 20
mobilenet_epoch = 20

**CNN**

**VGG**

In [ ]:
vgg16_model = VGG16(pooling='avg', weights='imagenet', include_top=False, input_shape=(224,224,3))
for layer in vgg16_model.layers:
    layer.trainable = False
last_output = vgg16_model.layers[-1].output
vgg_x = Flatten()(last_output)
vgg_x = Dense(512, activation='relu')(vgg_x)
vgg_x = Dense(256, activation='relu')(vgg_x)
vgg_x = Dense(128, activation='relu')(vgg_x)
vgg_x = Dense(15, activation='softmax')(vgg_x)
vgg16_final_model = Model(vgg16_model.input, vgg_x)
vgg16_final_model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

In [ ]:
# VGG16
number_of_epochs = vgg_epoch
vgg16_filepath = 'vgg_16_' + '-saved-model-{epoch:02d}-accuracy-{accuracy:.2f}.hdf5'
vgg_checkpoint = ModelCheckpoint(vgg16_filepath, monitor='accuracy', verbose=1, save_best_only=True, mode='max')
vgg_early_stopping = EarlyStopping(monitor='loss', patience=5)
vgg_reduce_lr_callback = ReduceLRonDrop(factor=0.2, threshold=0.05, patience=3, optimizer=optimizer)
vgg16_history = vgg16_final_model.fit(train_generator, epochs=number_of_epochs, callbacks=[vgg_reduce_lr_callback, vgg_checkpoint, vgg_early_stopping], verbose=1)

In [ ]:
do_history_stuff(vgg16_history, 'vgg16_model')
vgg16_final_model.save('VGG_model.keras')

**RES-NET-50**

In [ ]:
# ============================================
# ResNet50 Model Configuration and Training
# ============================================
ResNet50_model = ResNet50(input_shape=(224, 224, 3), weights='imagenet', include_top=False)

# Fine-tune all layers
for layer in ResNet50_model.layers:
    layer.trainable = True

ResNet50_last_output = ResNet50_model.output
ResNet50_maxpooled_output = Flatten()(ResNet50_last_output)
ResNet50_x = Dense(512, activation='relu')(ResNet50_maxpooled_output)
ResNet50_x = Dense(1024, activation='relu')(ResNet50_x)
ResNet50_x = Dropout(0.5)(ResNet50_x)
ResNet50_x = Dense(15, activation='softmax')(ResNet50_x)

ResNet50_x_final_model = Model(inputs=ResNet50_model.input, outputs=ResNet50_x)

number_of_epochs = resnet_epoch
resnet_filepath = 'resnet50_' + '-saved-model-{epoch:02d}-loss-{loss:.2f}.hdf5'

resnet_reduce_lr_callback = ReduceLRonDrop(factor=0.2, threshold=0.05, patience=3, optimizer=optimizer)
ResNet50_x_final_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
resnet_history = ResNet50_x_final_model.fit(train_generator, epochs=number_of_epochs, callbacks=[resnet_reduce_lr_callback], verbose=1)

In [ ]:
do_history_stuff(resnet_history, 'resnet50_model', True)
ResNet50_x_final_model.save('ResNet50.keras')

**MobileNetV2**

In [ ]:
MobileNetV2_model = MobileNetV2(input_shape=(224, 224, 3), weights='imagenet', include_top=False)

for layer in MobileNetV2_model.layers:
    layer.trainable = True

MobileNetV2_last_output = MobileNetV2_model.output
MobileNetV2_maxpooled_output = Flatten()(MobileNetV2_last_output)
MobileNetV2_x = Dense(512, activation='relu')(MobileNetV2_maxpooled_output)
MobileNetV2_x = Dense(1024, activation='relu')(MobileNetV2_x)
MobileNetV2_x = Dropout(0.5)(MobileNetV2_x)
MobileNetV2_x = Dense(15, activation='softmax')(MobileNetV2_x)

MobileNetV2_x_final_model = Model(inputs=MobileNetV2_model.input, outputs=MobileNetV2_x)

MobileNetV2_x_final_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

number_of_epochs = mobilenet_epoch
mobilenet_filepath = 'mobilenetv2_' + '-saved-model-{epoch:02d}-accuracy-{accuracy:.2f}.hdf5'

mobilenet_checkpoint = ModelCheckpoint(mobilenet_filepath, monitor='accuracy', verbose=1, save_best_only=True, mode='max')
mobilenet_early_stopping = EarlyStopping(monitor='loss', patience=5)
mobilenet_reduce_lr_callback = ReduceLRonDrop(factor=0.2, threshold=0.05, patience=3, optimizer=optimizer)

mobilenet_history = MobileNetV2_x_final_model.fit(train_generator, epochs=number_of_epochs, callbacks=[mobilenet_reduce_lr_callback, mobilenet_checkpoint, mobilenet_early_stopping], verbose=1)

In [ ]:
do_history_stuff(mobilenet_history, 'mobilenetv2_model', True)
MobileNetV2_x_final_model.save('MobileNetV2.keras')

**InceptionV3**

In [ ]:
Xception_model = InceptionV3(input_shape=(224, 224, 3), weights='imagenet', include_top=False)

for layer in Xception_model.layers:
    layer.trainable = True

Xception_last_output = Xception_model.output
Xception_maxpooled_output = Flatten()(Xception_last_output)
Xception_x = Dense(512, activation='relu')(Xception_maxpooled_output)
Xception_x = Dense(1024, activation='relu')(Xception_x)
Xception_x = Dropout(0.5)(Xception_x)
Xception_x = Dense(15, activation='softmax')(Xception_x)

Xception_x_final_model = Model(inputs=Xception_model.input, outputs=Xception_x)

Xception_x_final_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

number_of_epochs = mobilenet_epoch
xception_filepath = 'xception_' + '-saved-model-{epoch:02d}-accuracy-{accuracy:.2f}.hdf5'

xception_checkpoint = ModelCheckpoint(xception_filepath, monitor='accuracy', verbose=1, save_best_only=True, mode='max')
xception_early_stopping = EarlyStopping(monitor='loss', patience=5)
xception_reduce_lr_callback = ReduceLRonDrop(factor=0.2, threshold=0.05, patience=3, optimizer=optimizer)

xception_history = Xception_x_final_model.fit(train_generator, epochs=number_of_epochs, callbacks=[xception_reduce_lr_callback, xception_checkpoint, xception_early_stopping], verbose=1)

In [ ]:
do_history_stuff(xception_history, 'xception_model', True)
Xception_x_final_model.save('Xception.keras')

**AFTER TRAINING**

In [ ]:
vgg_best_model = vgg16_final_model
resnet_best_model = ResNet50_x_final_model
Xception_best_model = Xception_x_final_model
MobileNetV2_best_model = MobileNetV2_x_final_model

# plot on train

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

precision_scores = []
recall_scores = []
f1_scores = []

best_models = [vgg_best_model, resnet_best_model, Xception_best_model, MobileNetV2_best_model]

for model in best_models:
    train_preds = model.predict(train_generator)
    train_preds_labels = np.argmax(train_preds, axis=1)

    precision = precision_score(train_generator.classes, train_preds_labels, average='weighted') * 100
    recall = recall_score(train_generator.classes, train_preds_labels, average='weighted') * 100
    f1 = f1_score(train_generator.classes, train_preds_labels, average='weighted') * 100

    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

model_names = ['VGG', 'ResNet', 'Xception', 'MobileNet']

metrics_data = pd.DataFrame({'Model': model_names, 'Precision': precision_scores, 'Recall': recall_scores, 'F1 Score': f1_scores})
metrics_data = pd.melt(metrics_data, id_vars=['Model'], value_vars=['Precision', 'Recall', 'F1 Score'])

plt.figure(figsize=(12, 10))
sns.barplot(x='Model', y='value', hue='variable', data=metrics_data, palette='viridis_r')
plt.xlabel('Model', fontsize=18)
plt.ylabel('Score', fontsize=18)
plt.legend(title='Metrics', title_fontsize='12', fontsize='10')
plt.xticks(rotation=45)
plt.savefig('training_metrics_comparison.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
"""
Load the pre-trained models if they exist
"""
try:
    vgg_best_model = load_model('VGG_model.keras')
    print("✓ VGG model loaded successfully")
except Exception as e:
    print(f"⚠ VGG model not found: {e}")
    vgg_best_model = vgg16_final_model

try:
    resnet_best_model = load_model('ResNet50.keras')
    print("✓ ResNet50 model loaded successfully")
except Exception as e:
    print(f"⚠ ResNet50 model not found: {e}")
    resnet_best_model = ResNet50_x_final_model

try:
    Xception_best_model = load_model('Xception.keras')
    print("✓ Xception model loaded successfully")
except Exception as e:
    print(f"⚠ Xception model not found: {e}")
    Xception_best_model = Xception_x_final_model

try:
    MobileNetV2_best_model = load_model('MobileNetV2.keras')
    print("✓ MobileNetV2 model loaded successfully")
except Exception as e:
    print(f"⚠ MobileNetV2 model not found: {e}")
    MobileNetV2_best_model = MobileNetV2_x_final_model

In [ ]:
best_models = [vgg_best_model,resnet_best_model,Xception_best_model,MobileNetV2_best_model]

In [ ]:
"""
Plot Confusion Matrix for model evaluation
"""
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(y_true, y_pred, classes,
                          normalize=False,
                          title=None,
                          cmap=plt.cm.Blues):
    """
    Plot the confusion matrix.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
        classes: List of class labels
        normalize: Whether to normalize the confusion matrix
        title: Title for the plot
        cmap: Colormap to use
    """
    if not title:
        title = 'Normalized confusion matrix' if normalize else 'Confusion matrix'

    cm = confusion_matrix(y_true, y_pred)
    classes = [str(c) for c in classes]

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           xticklabels=classes, yticklabels=classes,
           title=title,
           ylabel='True label',
           xlabel='Predicted label')

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()

    return ax

# Define class labels
class_labels = list(range(15))
model_names = ['VGG', 'ResNet', 'Xception', 'MobileNet']

# Compute and plot confusion matrices for each model on training data
for idx, model in enumerate([vgg_best_model, resnet_best_model, Xception_best_model, MobileNetV2_best_model]):
    train_preds = model.predict(train_generator)
    train_preds_labels = np.argmax(train_preds, axis=1)
    
    plt.figure(figsize=(8, 6))
    plot_confusion_matrix(train_generator.classes, train_preds_labels, classes=class_labels, normalize=True)
    plt.savefig(f'confusion_matrix_{model_names[idx]}.png', dpi=600, bbox_inches='tight')
    plt.close()

print("Confusion matrices saved successfully!")

# ensembled

In [ ]:
"""
Helper function for ensemble voting
"""
def mode(my_list):
    """
    Returns the most common element(s) in a list
    
    Args:
        my_list: Input list
    
    Returns:
        List of most common element(s)
    """
    ct = Counter(my_list)
    max_value = max(ct.values())
    return [key for key, value in ct.items() if value == max_value]

In [ ]:
"""
Ensemble Model Prediction using Voting Strategy
Combines predictions from VGG, ResNet, Xception, and MobileNet
"""
true_value = []
combined_model_pred = []
vgg_pred = []
resnet_pred = []
xception_pred = []
mobilenetv2_pred = []

# Get number of batches from test generator
num_batches = len(test_generator)

print(f"Processing {num_batches} batches from test generator...")

for i in range(num_batches):
    img, true_class = test_generator[i]
    true_value.extend(np.argmax(true_class, axis=1))

    vgg_batch_pred = np.argmax(vgg_best_model.predict(img, verbose=0), axis=1)
    resnet_batch_pred = np.argmax(resnet_best_model.predict(img, verbose=0), axis=1)
    xception_batch_pred = np.argmax(Xception_best_model.predict(img, verbose=0), axis=1)
    mobilenetv2_batch_pred = np.argmax(MobileNetV2_best_model.predict(img, verbose=0), axis=1)

    vgg_pred.extend(vgg_batch_pred.tolist())
    resnet_pred.extend(resnet_batch_pred.tolist())
    xception_pred.extend(xception_batch_pred.tolist())
    mobilenetv2_pred.extend(mobilenetv2_batch_pred.tolist())

    batch_predictions = [
        resnet_best_model.predict(img, verbose=0),
        MobileNetV2_best_model.predict(img, verbose=0),
        Xception_best_model.predict(img, verbose=0),
        vgg_best_model.predict(img, verbose=0)
    ]
    ensemble_pred = np.argmax(np.mean(batch_predictions, axis=0), axis=1)
    combined_model_pred.extend(ensemble_pred.tolist())

# Calculate individual model accuracies
accuracy_vgg = sum(1 for p, t in zip(vgg_pred, true_value) if p == t) / len(true_value) * 100
accuracy_resnet = sum(1 for p, t in zip(resnet_pred, true_value) if p == t) / len(true_value) * 100
accuracy_xception = sum(1 for p, t in zip(xception_pred, true_value) if p == t) / len(true_value) * 100
accuracy_mobilenetv2 = sum(1 for p, t in zip(mobilenetv2_pred, true_value) if p == t) / len(true_value) * 100

# Calculate ensemble model accuracy
accuracy_combined = sum(1 for p, t in zip(combined_model_pred, true_value) if p == t) / len(true_value) * 100

print("\n" + "="*50)
print("MODEL ACCURACY RESULTS (Test Set)")
print("="*50)
print(f"VGG Accuracy:         {accuracy_vgg:.2f}%")
print(f"ResNet Accuracy:      {accuracy_resnet:.2f}%")
print(f"Xception Accuracy:    {accuracy_xception:.2f}%")
print(f"MobileNetV2 Accuracy: {accuracy_mobilenetv2:.2f}%")
print(f"Ensemble Accuracy:    {accuracy_combined:.2f}%")
print("="*50)

# stacking

In [ ]:
"""
Stacking: Collect predictions from base models for meta-learner training
"""
mobilenetv2_base_pred = []
resnet_base_pred = []
xception_base_pred = []
vgg_base_pred = []

print(f"Collecting base model predictions for stacking...")

for _ in range(len(test_generator)):
    batch_images, _ = next(test_generator)
    
    mobilenetv2_base_pred.append(np.argmax(MobileNetV2_best_model.predict(batch_images, verbose=0), axis=1))
    resnet_base_pred.append(np.argmax(resnet_best_model.predict(batch_images, verbose=0), axis=1))
    xception_base_pred.append(np.argmax(Xception_best_model.predict(batch_images, verbose=0), axis=1))
    vgg_base_pred.append(np.argmax(vgg_best_model.predict(batch_images, verbose=0), axis=1))

mobilenetv2_base_pred = np.concatenate(mobilenetv2_base_pred)
resnet_base_pred = np.concatenate(resnet_base_pred)
xception_base_pred = np.concatenate(xception_base_pred)
vgg_base_pred = np.concatenate(vgg_base_pred)

print("✓ Base model predictions collected successfully")

In [ ]:
"""
Alternative Ensemble Method: Hard Voting
Combines predictions using bincount (majority voting)
"""
# Convert lists to arrays if needed
resnet_pred_array = np.array(resnet_base_pred if len(resnet_base_pred) > 0 else resnet_pred)
mobilenetv2_pred_array = np.array(mobilenetv2_base_pred if len(mobilenetv2_base_pred) > 0 else mobilenetv2_pred)
xception_pred_array = np.array(xception_base_pred if len(xception_base_pred) > 0 else xception_pred)
vgg_pred_array = np.array(vgg_base_pred if len(vgg_base_pred) > 0 else vgg_pred)

# Combine predictions via hard voting
combined_predictions = np.column_stack((resnet_pred_array, mobilenetv2_pred_array, xception_pred_array, vgg_pred_array))

final_predictions = [np.argmax(np.bincount(sample_predictions)) for sample_predictions in combined_predictions]

accuracy_voting = sum(1 for p, t in zip(final_predictions, true_value) if p == t) / len(true_value) * 100

print(f"Hard Voting Ensemble Accuracy: {accuracy_voting:.2f}%")

# side by side

In [ ]:
"""
Load training histories from saved JSON files
"""
import pandas as pd

def load_history(file_name):
    """
    Load model training history from JSON file
    
    Args:
        file_name: Name of the file (without extension)
    
    Returns:
        DataFrame with training history or None if file doesn't exist
    """
    try:
        with open(file_name + '_history.json', 'r') as f:
            history_data = pd.read_json(f)
        return history_data
    except FileNotFoundError:
        print(f"⚠ History file not found: {file_name}_history.json")
        return None

# Load histories for each model
vgg16_history = load_history('vgg16_model')
resnet50_history = load_history('resnet50_model')
xception_history = load_history('xception_model')
mobilenetv2_history = load_history('mobilenetv2_model')

print("✓ Histories loaded successfully")

In [ ]:
true_value = test_generator.labels

In [ ]:
"""
Detailed Metrics Comparison (Precision, Recall, F1-Score)
"""
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

# Model names
model_names = ['VGG', 'ResNet', 'Xception', 'MobileNet']

# Initialize lists to store metrics
precision_scores = []
recall_scores = []
f1_scores = []

# Calculate metrics for each model's predictions
# Note: Raw scores are multiplied by (100/11.5) ≈ 8.7 for visibility in visualization
for model_preds in [vgg_pred, resnet_pred, xception_pred, mobilenetv2_pred]:
    # Handle potential mismatches in prediction array lengths
    valid_preds = model_preds[:len(true_value)]
    
    precision = precision_score(true_value, valid_preds, average='weighted', zero_division=0)
    recall = recall_score(true_value, valid_preds, average='weighted', zero_division=0)
    f1 = f1_score(true_value, valid_preds, average='weighted', zero_division=0)
    
    precision_scores.append(precision * 100)
    recall_scores.append(recall * 100)
    f1_scores.append(f1 * 100)

# Create DataFrame for visualization
metrics_data = pd.DataFrame({
    'Model': model_names, 
    'Precision': precision_scores, 
    'Recall': recall_scores, 
    'F1 Score': f1_scores
})

# Melt for grouped bar plot
metrics_melted = pd.melt(metrics_data, id_vars=['Model'], 
                         value_vars=['Precision', 'Recall', 'F1 Score'])

# Create visualization
plt.figure(figsize=(12, 6))
sns.barplot(x='Model', y='value', hue='variable', data=metrics_melted, palette='viridis_r')
plt.xlabel('Model', fontsize=14)
plt.ylabel('Score (%)', fontsize=14)
plt.title('Precision, Recall, and F1 Score Comparison', fontsize=16)
plt.legend(title='Metrics', title_fontsize='12', fontsize='11')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=600, bbox_inches='tight')
plt.show()

print("\n" + "="*50)
print("DETAILED METRICS (Test Set)")
print("="*50)
print(metrics_data.to_string(index=False))
print("="*50)

# acc

In [ ]:
"""
Plot Accuracy Comparison Across Epochs
"""
import seaborn as sns
import matplotlib.pyplot as plt

def plot_accuracy_comparison(histories, model_names, filename):
    """
    Plot accuracy across epochs for multiple models
    
    Args:
        histories: List of history DataFrames
        model_names: List of model names
        filename: Output filename for saving the plot
    """
    sns.set(style="whitegrid")
    plt.figure(figsize=(12, 7))
    
    custom_colors = ['blue', 'green', 'red', 'purple']
    
    for i, history in enumerate(histories):
        if history is None:
            print(f"⚠ Skipping {model_names[i]} - history not available")
            continue
            
        model_name = model_names[i]
        color = custom_colors[i % len(custom_colors)]
        
        acc_key = 'accuracy' if 'accuracy' in history.history else ('acc' if 'acc' in history.history else None)
        
        if acc_key is None:
            print(f"⚠ No accuracy key found in {model_name} history")
            continue
        
        plt.plot(range(len(history.history[acc_key])), history.history[acc_key], 
                label=f'{model_name} Training Accuracy', color=color, marker='o', markersize=5)
    
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('Accuracy', fontsize=14)
    plt.title('Training Accuracy Comparison Across Models', fontsize=16)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(filename, dpi=600, bbox_inches='tight', format='png')
    plt.show()

# Plot accuracy for all models
histories = [vgg16_history, resnet50_history, xception_history, mobilenetv2_history]
plot_accuracy_comparison(histories, ['VGG16', 'ResNet50', 'Xception', 'MobileNetV2'], 
                        'accuracy_plot_custom_colors.png')

# loss

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_loss_comparison(histories, model_names, filename):
    sns.set(style="whitegrid")
    plt.figure(figsize=(12, 10))
    
    custom_colors = ['blue', 'green', 'red', 'purple', 'orange', 'brown', 'pink', 'gray', 'cyan']
    
    for i, history in enumerate(histories):
        if history is None:
            print(f"⚠ Skipping {model_names[i]} - history not available")
            continue
        model_name = model_names[i]
        color = custom_colors[i % len(custom_colors)]
        
        loss_key = 'loss'
        plt.plot(range(len(history.history[loss_key])), history.history[loss_key], label=f'{model_name} Training Loss', color=color, marker='o', markersize=6)
    
    plt.xlabel('Epoch', fontsize=18)
    plt.ylabel('Loss', fontsize=18)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=600, bbox_inches='tight', format='png')
    plt.show()

# Plot and save loss for all models with customized line colors
plot_loss_comparison([vgg16_history, resnet50_history, xception_history, mobilenetv2_history],
                     ['VGG16', 'ResNet50', 'Xception', 'MobileNetV2'], 'loss_plot_custom_colors.png')